# DL Masterclass 05: Regularization & Normalization

Welcome to Notebook 05. A Deep Neural Network is an incredible memorization machine. If you give a 100-layer network a dataset of 1,000 random noise images and completely random labels, it will eventually achieve 100% Training Accuracy by perfectly memorizing every pixel. 

To prevent memorization and force *generalization*, we must intentionally sabotage the network during training.

---

## 🎓 Socratic Deep Check
Ponder this question before we begin. It addresses how PyTorch hides complex mathematics from you in `model.eval()`:

> *"Dropout sets 50% of node activations to exactly 0.0 during training. During production inference, we turn Dropout off. If a downstream Layer was used to receiving a sum of $50$ from 50 active nodes, it will suddenly receive a sum of $100$ from 100 active nodes during inference, completely blowing up the math. How does PyTorch secretly fix this behind the scenes?"*

---

## Table of Contents
1. **Dropout:** The mathematics of Inverted Dropout and scaling.
2. **Internal Covariate Shift:** Why deep networks destabilize internally.
3. **Batch Normalization:** The algorithm, and its fatal flaw in production.
4. **Layer Normalization:** Why Transformers abandoned Batch Norm.


## 1. Dropout (Inverted Scale)

Dropout randomly zeroes out a percentage (e.g., $p=0.5$) of neurons during each forward pass. This prevents 'co-adaptation' (where neuron A relies entirely on neuron B to fix its mistakes). It forces the network to learn redundant, robust feature representations.

### 🎓 DEEP QUESTION ANSWERED
**Q:** *How does PyTorch solve the scale mismatch at Inference?*

**A:** Originally (in 2012), researchers would multiply all weights by $p$ during Inference to scale the signal *down*.

However, modern frameworks (PyTorch/Tensorflow) use **Inverted Dropout**. 
During *Training*, right after they zero out 50% of the nodes, they take the remaining active nodes and intentionally **divide by $(1 - p)$** (e.g., multiplying by 2). 

This scales the training signal *up* artificially. 
Why? Because it means that during Inference, when $100\%$ of the nodes are active, you literally have to do **nothing**. The expected value of the sum remains mathematically identical. No scaling is needed in production APIs, making them faster.

In [ ]:
import numpy as np

# -----------------------------------------------------
# IMPLEMENTATION: Inverted Dropout vs Naive Dropout
# -----------------------------------------------------
np.random.seed(42)

# 100 neurons, all outputting the value '1.0'
activations = np.ones(100)
p = 0.5 # Drop 50%

print("--- NAIVE DROPOUT (Old Way) ---")
mask = np.random.binomial(1, 1-p, size=activations.shape)
train_activations_naive = activations * mask
print(f"Training Expected Sum: {np.sum(train_activations_naive):.1f}")

# During inference, all nodes are on. Sum is 100.
# We must remember to scale by p:
inference_naive = np.sum(activations) * p
print(f"Inference (Calculated): {inference_naive:.1f}\n")


print("--- INVERTED DROPOUT (PyTorch Way) ---")
mask = np.random.binomial(1, 1-p, size=activations.shape)
# Scale it UP instantly!
train_activations_inverted = (activations * mask) / (1.0 - p)
print(f"Training Expected Sum: {np.sum(train_activations_inverted):.1f}")

# Inference requires absolutely zero math.
inference_inverted = np.sum(activations)
print(f"Inference (Raw Sum): {inference_inverted:.1f}")

# Note how the Training Sum matches the Inference Sum intrinsically.

## 2. Internal Covariate Shift

In Notebook 03, we solved the initialization problem for Layer 1. But what happens at Layer 50?

As Layer 1 updates its weights, its output distribution shifts slightly. Layer 2 receives this new distribution, updates its weights, and causes a slightly larger shift. By Layer 50, the distribution of the activations is flailing violently across the mathematical landscape. 

This is **Internal Covariate Shift**. It forces us to use tiny learning rates so the later layers aren't constantly having the "rug pulled out from under them" by the earlier layers.

## 3. Batch Normalization

Batch Normalization (2015) mathematically enforces that the output of every single layer has a Mean of exactly 0 and a Variance of exactly 1, no matter what the previous layers do.

$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}$

But enforcing exactly Mean=0 destroys the expressive power of the network. So they added two *learnable* parameters, $\gamma$ (scale) and $\beta$ (shift):

$y_i = \gamma \hat{x}_i + \beta$

**The Network learns exactly how much it wants to un-normalize the data.**

### The Fatal Flaw of Batch Norm
Batch Norm requires a *Batch* to calculate $\mu_B$ and $\sigma_B$. 
During Training, batch size is 64. No problem.
During Inference (Production API), batch size is usually 1 (a single user request). You cannot calculate the variance of 1 number (it is mathematically undefined).

**How PyTorch fixes it:** During training, PyTorch secretly maintains a hidden `running_mean` and `running_var` of every batch it has ever seen using Exponential Moving Averages. When you call `model.eval()`, it physically stops looking at the incoming data's mean, and hardcodes the historical `running_mean` into the equation.

## 4. Layer Normalization (The LLM Standard)

Because Batch Norm breaks down completely if the batch size is small (e.g., training massive models where only 2 images fit on the GPU), and because it is mathematically incompatible with Sequence Data (like varying-length sentences in NLP), the industry moved to **Layer Normalization**.

*   **Batch Norm:** Normalizes across the Batch ($N$ axis) for a single specific feature.
*   **Layer Norm:** Normalizes across the Features ($F$ axis) for a single specific data point.

Because Layer Norm normalizes *within* a single sample, it is 100% mathematically identical during both Training and Inference. Batch size is irrelevant. It is the core stabilizer of modern Transformer architectures.